# Reliability-qualified Haiyan nighttime lights: baseline, shock, and recovery

This notebook tests whether VNP46A2 can support a defensible estimate of electricity-mediated nighttime activity disruption and recovery after Haiyan in Samar–Leyte.

The workflow preserves the required order of inference:

1. **Observability:** determine when and where fresh high-quality DNB-BRDF observations exist.
2. **Interpretability:** retain pixels with adequate, non-zero, stable pre-event behaviour.
3. **Recovery:** estimate impact, T50, T80, and cumulative deficit only where observation support is sufficient.

`DNB_BRDF_Corrected_NTL` is the primary evidentiary signal. `Gap_Filled_DNB_BRDF_Corrected_NTL` is used only to diagnose how product filling changes the apparent trajectory. Missing observations are retained as missing; they are not silently interpolated.

## Analytical design

- **Long baseline:** 180 to 31 days before Haiyan; used to construct the pixel-level baseline.
- **Immediate baseline:** 90 to 8 days before Haiyan; used to assess short-term pre-event stability.
- **Exclusion window:** 7 to 1 days before landfall; excluded from baseline estimation.
- **Fresh observation:** finite DNB-BRDF with `Mandatory_Quality_Flag == 0` and no snow flag.
- **Baseline-eligible pixel:** sufficient fresh baseline observations, baseline radiance above the low-signal threshold, and acceptable robust relative variability.
- **Daily profile:** median of each fresh pixel relative to its own baseline. This limits changes caused only by different pixels being visible on different dates.
- **T50/T80:** first sustained threshold crossing after the observed impact. Timing is reported as an interval between the last qualified observation below and the first sustained qualified observation above the threshold.

The current pre-event archive can complete the baseline, observability, anomaly, and spatial-context sections. Post-event stacks placed in the same directory automatically activate the impact and recovery sections.

In [1]:
from contextlib import ExitStack
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import rasterio

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from IPython.display import display, Markdown
from tqdm.auto import tqdm


/Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Paths
DATA_DIR = Path("../datasets/VNP46")
INPUT_PATTERN = "Haiyan_VNP46A2_AllBands_*"

INPUT_FILES = sorted(
    path
    for path in DATA_DIR.glob(INPUT_PATTERN)
    if path.suffix.lower() in {".tif", ".tiff"}
)

OUTPUT_DIR = Path("outputs/haiyan_vnp46a2_recovery")
OUTPUT_TABLE_DIR = OUTPUT_DIR / "tables"
OUTPUT_FIGURE_DIR = OUTPUT_DIR / "figures"

for directory in [OUTPUT_TABLE_DIR, OUTPUT_FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

if not INPUT_FILES:
    raise FileNotFoundError(
        f"No VNP46A2 GeoTIFFs match {DATA_DIR / INPUT_PATTERN}. "
        "Update DATA_DIR or INPUT_PATTERN."
    )


# Event and baseline settings
EVENT_DATE = pd.Timestamp("2013-11-08")
EXPECTED_START = pd.Timestamp("2013-05-12")
EXPECTED_END = pd.Timestamp("2014-11-08")

LONG_BASELINE_START = -180
LONG_BASELINE_END = -31
IMMEDIATE_BASELINE_START = -90
IMMEDIATE_BASELINE_END = -8

MIN_BASELINE_OBSERVATIONS = 10
MIN_BASELINE_RADIANCE = 0.5
MAX_BASELINE_ROBUST_RSD = 0.75
MIN_DAILY_COVERAGE = 20

SMOOTHING_WINDOW = "14D"
MIN_SMOOTHING_OBSERVATIONS = 2
IMPACT_WINDOW_DAYS = 30
MAX_INTEGRATION_GAP_DAYS = 14
RECOVERY_PERSISTENCE_DAYS = 14

# Earth Engine VNP46A2 Collection 2 exports are normally in physical values.
# Use "raw" only if the GeoTIFF contains unscaled product integers.
SCALING_MODE = "auto"  # "auto", "physical", or "raw"
NO_DATA = -9999.0
MAX_MAP_DIMENSION = 650

SAVE_OUTPUTS = True
SAVE_PNG = False


COLORS = {
    "observed": "#222222",
    "smoothed": "#d62728",
    "gap_filled": "#ff7f0e",
    "coverage": "#2ca02c",
    "cloud": "#7f7f7f",
    "event": "#1f77b4",
}

In [4]:
EXPECTED_BANDS = [
    "DNB_BRDF_Corrected_NTL",
    "Gap_Filled_DNB_BRDF_Corrected_NTL",
    "DNB_Lunar_Irradiance",
    "Latest_High_Quality_Retrieval",
    "Mandatory_Quality_Flag",
    "QF_Cloud_Mask",
    "Snow_Flag",
]

RADIANCE_BANDS = {
    "DNB_BRDF_Corrected_NTL",
    "Gap_Filled_DNB_BRDF_Corrected_NTL",
    "DNB_Lunar_Irradiance",
}

FILL_VALUES = {
    "DNB_BRDF_Corrected_NTL": [65535],
    "Gap_Filled_DNB_BRDF_Corrected_NTL": [65535],
    "DNB_Lunar_Irradiance": [65535],
    "Latest_High_Quality_Retrieval": [255],
    "Mandatory_Quality_Flag": [255],
    "QF_Cloud_Mask": [65535],
    "Snow_Flag": [255],
}


def parse_date(text):
    text = "" if text is None else str(text)

    calendar_match = re.search(
        r"((?:19|20)\d{2})[_-]?(\d{2})[_-]?(\d{2})",
        text,
    )
    if calendar_match:
        return pd.Timestamp(
            year=int(calendar_match.group(1)),
            month=int(calendar_match.group(2)),
            day=int(calendar_match.group(3)),
        )

    julian_match = re.search(r"A((?:19|20)\d{2})(\d{3})", text)
    if julian_match:
        return pd.to_datetime(
            f"{julian_match.group(1)}-{julian_match.group(2)}",
            format="%Y-%j",
        )

    return pd.NaT


def parse_band(description, filename, band_index):
    description = "" if description is None else str(description)
    variable = next(
        (
            band
            for band in sorted(EXPECTED_BANDS, key=len, reverse=True)
            if band in description
        ),
        None,
    )

    date = parse_date(description)
    if pd.isna(date):
        date = parse_date(filename)

    return {
        "band_index": band_index,
        "description": description,
        "Date": date,
        "Variable": variable,
    }


def create_inventory(input_files):
    band_rows = []
    file_rows = []

    for priority, path in enumerate(input_files):
        with rasterio.open(path) as src:
            file_rows.append({
                "File": str(path),
                "Priority": priority,
                "Width": src.width,
                "Height": src.height,
                "Bands": src.count,
                "CRS": src.crs.to_string() if src.crs else None,
                "Transform": tuple(src.transform)[:6],
                "Bounds": tuple(src.bounds),
                "Size_GB": path.stat().st_size / 1024**3,
            })

            for band_index in range(1, src.count + 1):
                description = src.descriptions[band_index - 1]
                if description is None:
                    tags = src.tags(band_index)
                    description = (
                        tags.get("DESCRIPTION")
                        or tags.get("description")
                        or tags.get("NAME")
                    )

                row = parse_band(description, path.name, band_index)
                row.update({
                    "File": str(path),
                    "Priority": priority,
                })
                band_rows.append(row)

    return pd.DataFrame(band_rows), pd.DataFrame(file_rows)


def decode_cloud_mask(values):
    integers = np.zeros(values.shape, dtype=np.uint16)
    valid = np.isfinite(values)
    integers[valid] = np.rint(values[valid]).astype(np.uint16)

    return {
        "valid": valid,
        "land_water": (integers >> 1) & 7,
        "mask_quality": (integers >> 4) & 3,
        "cloud_confidence": (integers >> 6) & 3,
        "snow_ice": (integers >> 10) & 1,
    }


def clean_values(values, variable, nodata, radiance_scale):
    values = values.astype(np.float32)
    invalid = ~np.isfinite(values) | np.isclose(values, NO_DATA)

    if nodata is not None:
        invalid |= np.isclose(values, nodata)

    for fill_value in FILL_VALUES[variable]:
        invalid |= np.isclose(values, fill_value)

    values[invalid] = np.nan

    if variable in RADIANCE_BANDS:
        values *= radiance_scale

    return values


def read_band(sources, lookup, date, variable, radiance_scale):
    key = (pd.Timestamp(date), variable)
    if key not in lookup:
        return None

    record = lookup[key]
    src = sources[record["File"]]
    values = src.read(int(record["band_index"]))

    return clean_values(
        values,
        variable,
        src.nodata,
        radiance_scale,
    )


def percentage(mask, denominator):
    denominator_count = np.count_nonzero(denominator)
    if denominator_count == 0:
        return np.nan
    return np.count_nonzero(mask & denominator) / denominator_count * 100


def masked_median(values, mask):
    selected = values[mask & np.isfinite(values)]
    return float(np.nanmedian(selected)) if selected.size else np.nan


def apply_figure_style(fig, width=1300, height=720, top=115, right=80, bottom=75):
    fig.update_layout(
        template="plotly_white",
        width=width,
        height=height,
        margin=dict(l=85, r=right, t=top, b=bottom),
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="white",
        font=dict(size=15),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1,
        ),
        hovermode="x unified",
    )
    fig.update_xaxes(showgrid=True, gridcolor="rgba(0,0,0,0.08)")
    fig.update_yaxes(showgrid=True, gridcolor="rgba(0,0,0,0.08)")


def add_haiyan_marker(fig, row=None, col=None):
    kwargs = {} if row is None else {"row": row, "col": col}
    fig.add_vline(
        x=EVENT_DATE,
        line_width=2,
        line_dash="dash",
        line_color=COLORS["event"],
        **kwargs,
    )


def export_figure(fig, name):
    if not SAVE_OUTPUTS:
        return
    fig.write_html(OUTPUT_FIGURE_DIR / f"{name}.html")
    if SAVE_PNG:
        fig.write_image(
            OUTPUT_FIGURE_DIR / f"{name}.png",
            scale=2,
        )

In [5]:
band_inventory, file_inventory = create_inventory(INPUT_FILES)

unparsed = band_inventory[
    band_inventory["Date"].isna()
    | band_inventory["Variable"].isna()
]

if not unparsed.empty:
    display(unparsed[["File", "band_index", "description"]].head(30))
    raise ValueError(
        "Some GeoTIFF bands could not be parsed. "
        "Inspect the displayed descriptions before continuing."
    )

grid_signatures = set(
    zip(
        file_inventory["Width"],
        file_inventory["Height"],
        file_inventory["CRS"],
        file_inventory["Transform"].astype(str),
        file_inventory["Bounds"].astype(str),
    )
)

if len(grid_signatures) != 1:
    raise ValueError("Input GeoTIFFs do not share one spatial grid.")

duplicate_count = int(
    band_inventory.duplicated(["Date", "Variable"], keep=False).sum()
)

selected_bands = (
    band_inventory
    .sort_values(["Date", "Variable", "Priority"])
    .drop_duplicates(["Date", "Variable"], keep="last")
)

band_lookup = {
    (row.Date, row.Variable): {
        "File": row.File,
        "band_index": row.band_index,
    }
    for row in selected_bands.itertuples()
}

date_inventory = (
    selected_bands
    .groupby("Date")["Variable"]
    .agg(lambda values: set(values))
    .rename("Available")
    .reset_index()
)
date_inventory["Missing"] = date_inventory["Available"].apply(
    lambda values: sorted(set(EXPECTED_BANDS) - values)
)
date_inventory["Complete"] = date_inventory["Missing"].str.len().eq(0)

available_dates = sorted(
    date_inventory.loc[date_inventory["Complete"], "Date"].tolist()
)

if not available_dates:
    raise ValueError("No date contains all seven required VNP46A2 bands.")

with rasterio.open(file_inventory.iloc[0]["File"]) as reference_src:
    native_height = reference_src.height
    native_width = reference_src.width
    native_transform = reference_src.transform
    native_crs = reference_src.crs

expected_dates = pd.date_range(EXPECTED_START, EXPECTED_END, freq="D")
missing_dates = expected_dates.difference(pd.DatetimeIndex(available_dates))

inventory_summary = pd.DataFrame([
    {"Item": "GeoTIFF stacks", "Value": len(INPUT_FILES)},
    {"Item": "Parsed bands", "Value": len(selected_bands)},
    {"Item": "Complete dates available", "Value": len(available_dates)},
    {"Item": "Expected calendar dates", "Value": len(expected_dates)},
    {"Item": "Dates not yet available", "Value": len(missing_dates)},
    {"Item": "Overlapping bands resolved", "Value": duplicate_count},
    {"Item": "First available date", "Value": min(available_dates).date()},
    {"Item": "Last available date", "Value": max(available_dates).date()},
])

display(inventory_summary)
display(file_inventory[["File", "Width", "Height", "Bands", "Size_GB"]].round(3))

if SAVE_OUTPUTS:
    date_inventory.assign(
        Missing=date_inventory["Missing"].apply(", ".join)
    ).to_csv(OUTPUT_TABLE_DIR / "vnp46a2_date_inventory.csv", index=False)

,Item,Value
0,GeoTIFF stacks,2
1,Parsed bands,2555
2,Complete dates available,365
3,Expected calendar dates,546
4,Dates not yet available,181
5,Overlapping bands resolved,0
6,First available date,2013-05-12
7,Last available date,2014-11-08


,File,Width,Height,Bands,Size_GB
0,../datasets/VNP46/Haiyan_VNP46A2_AllBands_Post...,473,674,1295,0.203
1,../datasets/VNP46/Haiyan_VNP46A2_AllBands_PreE...,473,674,1260,0.191


## Pixel-level baseline and daily profiles

Each eligible pixel is normalized by its own long-baseline median. The regional trajectory therefore compares the same pixel with its expected pre-event level before aggregating across the pixels visible on a given date.

The gap-filled profile uses the same pixel baseline but does not replace missing fresh observations in the primary trajectory.

In [ ]:
baseline_dates = [
    date
    for date in available_dates
    if LONG_BASELINE_START
    <= (date - EVENT_DATE).days
    <= LONG_BASELINE_END
]

if len(baseline_dates) < MIN_BASELINE_OBSERVATIONS:
    raise ValueError(
        f"Only {len(baseline_dates)} baseline dates are available; "
        f"at least {MIN_BASELINE_OBSERVATIONS} are required."
    )

# Detect whether radiance arrays are physical values or raw scaled integers.
with rasterio.open(band_lookup[(baseline_dates[0], "DNB_BRDF_Corrected_NTL")]["File"]) as src:
    sample_record = band_lookup[(baseline_dates[0], "DNB_BRDF_Corrected_NTL")]
    sample = src.read(int(sample_record["band_index"])).astype(np.float32)
sample = sample[
    np.isfinite(sample)
    & ~np.isclose(sample, NO_DATA)
    & ~np.isclose(sample, 65535)
]

sample_p99 = float(np.nanpercentile(sample, 99)) if sample.size else np.nan
sample_integer_fraction = (
    float(np.mean(np.isclose(sample, np.rint(sample))))
    if sample.size
    else np.nan
)

if SCALING_MODE == "raw":
    radiance_scale = 0.1
elif SCALING_MODE == "physical":
    radiance_scale = 1.0
else:
    appears_raw = (
        sample_p99 > 500
        or (
            sample_p99 > 20
            and sample_integer_fraction > 0.98
        )
    )
    radiance_scale = 0.1 if appears_raw else 1.0

scaling_label = (
    "raw product integers × 0.1"
    if radiance_scale == 0.1
    else "physical radiance values"
)

baseline_layers = []

with ExitStack() as stack:
    sources = {
        str(path): stack.enter_context(rasterio.open(path))
        for path in INPUT_FILES
    }

    for date in tqdm(baseline_dates, desc="Constructing baseline"):
        dnb = read_band(
            sources, band_lookup, date,
            "DNB_BRDF_Corrected_NTL", radiance_scale,
        )
        mqf = read_band(
            sources, band_lookup, date,
            "Mandatory_Quality_Flag", radiance_scale,
        )
        snow = read_band(
            sources, band_lookup, date,
            "Snow_Flag", radiance_scale,
        )
        cloud_qf = read_band(
            sources, band_lookup, date,
            "QF_Cloud_Mask", radiance_scale,
        )
        cloud = decode_cloud_mask(cloud_qf)
        land = cloud["valid"] & np.isin(cloud["land_water"], [0, 1, 5])

        fresh = (
            land
            & np.isfinite(dnb)
            & np.isfinite(mqf)
            & (np.rint(mqf) == 0)
            & np.isfinite(snow)
            & (np.rint(snow) == 0)
        )

        baseline_layers.append(np.where(fresh, dnb, np.nan).astype(np.float32))

    baseline_stack = np.stack(baseline_layers)
    baseline_count = np.sum(np.isfinite(baseline_stack), axis=0)
    baseline_median = np.nanmedian(baseline_stack, axis=0)
    baseline_mad = np.nanmedian(
        np.abs(baseline_stack - baseline_median[None, :, :]),
        axis=0,
    )
    baseline_robust_rsd = np.divide(
        1.4826 * baseline_mad,
        baseline_median,
        out=np.full_like(baseline_median, np.nan, dtype=np.float32),
        where=baseline_median > 0,
    )

    eligible_mask = (
        (baseline_count >= MIN_BASELINE_OBSERVATIONS)
        & np.isfinite(baseline_median)
        & (baseline_median >= MIN_BASELINE_RADIANCE)
        & np.isfinite(baseline_robust_rsd)
        & (baseline_robust_rsd <= MAX_BASELINE_ROBUST_RSD)
    )

    del baseline_stack, baseline_layers

    eligible_count = np.count_nonzero(eligible_mask)
    if eligible_count == 0:
        raise ValueError(
            "No pixel satisfies the baseline criteria. Review the scaling "
            "and baseline thresholds before continuing."
        )

    daily_rows = []

    for date in tqdm(available_dates, desc="Daily VNP46A2 profiles"):
        arrays = {
            variable: read_band(
                sources, band_lookup, date, variable, radiance_scale
            )
            for variable in EXPECTED_BANDS
        }

        cloud = decode_cloud_mask(arrays["QF_Cloud_Mask"])
        land = cloud["valid"] & np.isin(cloud["land_water"], [0, 1, 5])

        fresh = (
            eligible_mask
            & land
            & np.isfinite(arrays["DNB_BRDF_Corrected_NTL"])
            & np.isfinite(arrays["Mandatory_Quality_Flag"])
            & (np.rint(arrays["Mandatory_Quality_Flag"]) == 0)
            & np.isfinite(arrays["Snow_Flag"])
            & (np.rint(arrays["Snow_Flag"]) == 0)
        )

        gap_valid = (
            eligible_mask
            & land
            & np.isfinite(arrays["Gap_Filled_DNB_BRDF_Corrected_NTL"])
        )
        gap_only = gap_valid & ~fresh
        cloudy = eligible_mask & land & (cloud["cloud_confidence"] >= 2)
        snow_affected = (
            eligible_mask
            & np.isfinite(arrays["Snow_Flag"])
            & (np.rint(arrays["Snow_Flag"]) == 1)
        )

        observed_relative = np.divide(
            arrays["DNB_BRDF_Corrected_NTL"],
            baseline_median,
            out=np.full_like(baseline_median, np.nan, dtype=np.float32),
            where=fresh,
        ) * 100

        gap_relative = np.divide(
            arrays["Gap_Filled_DNB_BRDF_Corrected_NTL"],
            baseline_median,
            out=np.full_like(baseline_median, np.nan, dtype=np.float32),
            where=gap_valid,
        ) * 100

        daily_rows.append({
            "Date": date,
            "Days_from_event": (date - EVENT_DATE).days,
            "Fresh_coverage_pct": percentage(fresh, eligible_mask),
            "Fresh_pixels": int(np.count_nonzero(fresh)),
            "Relative_DNB_pct": masked_median(observed_relative, fresh),
            "Relative_gap_filled_pct": masked_median(gap_relative, gap_valid),
            "Gap_filled_only_pct": percentage(gap_only, eligible_mask),
            "Cloudy_pct": percentage(cloudy, eligible_mask),
            "Snow_pct": percentage(snow_affected, eligible_mask),
            "Retrieval_age_median": masked_median(
                arrays["Latest_High_Quality_Retrieval"], gap_valid
            ),
            "Lunar_irradiance_median": masked_median(
                arrays["DNB_Lunar_Irradiance"], fresh
            ),
        })

daily_observed = pd.DataFrame(daily_rows).sort_values("Date")
daily_df = (
    pd.DataFrame({"Date": expected_dates})
    .merge(daily_observed, on="Date", how="left")
)
daily_df["Days_from_event"] = (daily_df["Date"] - EVENT_DATE).dt.days
daily_df["File_available"] = daily_df["Date"].isin(available_dates)
daily_df["Qualified"] = (
    daily_df["File_available"]
    & daily_df["Relative_DNB_pct"].notna()
    & daily_df["Fresh_coverage_pct"].ge(MIN_DAILY_COVERAGE)
)
daily_df["Gap_available"] = (
    daily_df["File_available"]
    & daily_df["Relative_gap_filled_pct"].notna()
)

qualified_series = (
    daily_df.set_index("Date")["Relative_DNB_pct"]
    .where(daily_df.set_index("Date")["Qualified"])
)
daily_df["Relative_DNB_smoothed_pct"] = (
    qualified_series
    .rolling(
        SMOOTHING_WINDOW,
        center=True,
        min_periods=MIN_SMOOTHING_OBSERVATIONS,
    )
    .median()
    .reindex(daily_df["Date"])
    .to_numpy()
)

daily_df["Relative_gap_smoothed_pct"] = (
    daily_df.set_index("Date")["Relative_gap_filled_pct"]
    .rolling(
        SMOOTHING_WINDOW,
        center=True,
        min_periods=MIN_SMOOTHING_OBSERVATIONS,
    )
    .median()
    .reindex(daily_df["Date"])
    .to_numpy()
)


def longest_calendar_gap(dates, phase_start, phase_end):
    observed = pd.DatetimeIndex(sorted(pd.to_datetime(dates)))
    boundaries = pd.DatetimeIndex([
        phase_start - pd.Timedelta(days=1),
        *observed,
        phase_end + pd.Timedelta(days=1),
    ])
    return int(np.diff(boundaries.values).astype("timedelta64[D]").astype(int).max() - 1)


phase_definitions = [
    ("Long baseline", EVENT_DATE + pd.Timedelta(days=LONG_BASELINE_START),
     EVENT_DATE + pd.Timedelta(days=LONG_BASELINE_END)),
    ("Shock", EVENT_DATE, EVENT_DATE + pd.Timedelta(days=IMPACT_WINDOW_DAYS)),
    ("Recovery", EVENT_DATE + pd.Timedelta(days=IMPACT_WINDOW_DAYS + 1), EXPECTED_END),
]

phase_rows = []
for phase, phase_start, phase_end in phase_definitions:
    phase_data = daily_df[daily_df["Date"].between(phase_start, phase_end)]
    qualified_dates = phase_data.loc[phase_data["Qualified"], "Date"]
    phase_rows.append({
        "Phase": phase,
        "Calendar_days": len(phase_data),
        "Files_available": int(phase_data["File_available"].sum()),
        "Qualified_days": int(phase_data["Qualified"].sum()),
        "Median_fresh_coverage_pct": phase_data["Fresh_coverage_pct"].median(),
        "Longest_qualified_gap_days": longest_calendar_gap(
            qualified_dates, phase_start, phase_end
        ),
        "First_qualified_date": (
            qualified_dates.min().date()
            if not qualified_dates.empty
            else pd.NaT
        ),
    })

phase_support_df = pd.DataFrame(phase_rows)

baseline_summary = pd.DataFrame([
    {"Criterion": "Radiance scaling", "Value": scaling_label},
    {"Criterion": "Baseline dates available", "Value": len(baseline_dates)},
    {"Criterion": "Eligible pixels", "Value": eligible_count},
    {
        "Criterion": "Eligible share of raster",
        "Value": f"{eligible_count / baseline_median.size * 100:.2f}%",
    },
    {
        "Criterion": "Median eligible baseline radiance",
        "Value": f"{np.nanmedian(baseline_median[eligible_mask]):.3f}",
    },
    {
        "Criterion": "Median eligible robust RSD",
        "Value": f"{np.nanmedian(baseline_robust_rsd[eligible_mask]):.3f}",
    },
])

display(baseline_summary)
display(phase_support_df.round(2))
display(daily_df.loc[daily_df["File_available"]].head().round(3))

if SAVE_OUTPUTS:
    baseline_summary.to_csv(
        OUTPUT_TABLE_DIR / "baseline_summary.csv", index=False
    )
    daily_df.to_csv(
        OUTPUT_TABLE_DIR / "daily_reliability_qualified_profiles.csv",
        index=False,
    )
    phase_support_df.to_csv(
        OUTPUT_TABLE_DIR / "phase_observation_support.csv",
        index=False,
    )

Constructing baseline: 100%|██████████| 150/150 [29:08<00:00, 11.66s/it]
/var/folders/91/f5wz25_x3r586p828vt1hm080000gp/T/ipykernel_20293/2829679959.py:93: RuntimeWarning: All-NaN slice encountered
  baseline_median = np.nanmedian(baseline_stack, axis=0)
/var/folders/91/f5wz25_x3r586p828vt1hm080000gp/T/ipykernel_20293/2829679959.py:94: RuntimeWarning: All-NaN slice encountered
  baseline_mad = np.nanmedian(
Daily VNP46A2 profiles:   1%|          | 2/365 [00:50<2:32:18, 25.18s/it]

## Observability and nighttime lights trajectory

The upper panel shows the fresh DNB-BRDF trajectory, its short-window median, and the gap-filled diagnostic. The lower panel shows how much baseline-eligible area is freshly observed and how much is classified as cloudy. Dates below the coverage threshold remain visible but do not contribute to recovery metrics.

In [1]:
fig_trajectory = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.10,
    row_heights=[0.68, 0.32],
)

fig_trajectory.add_trace(
    go.Scatter(
        x=daily_df["Date"],
        y=daily_df["Relative_DNB_pct"],
        mode="markers",
        name="Fresh DNB-BRDF",
        marker=dict(
            size=7,
            color=np.where(
                daily_df["Qualified"],
                COLORS["observed"],
                "rgba(120,120,120,0.35)",
            ),
        ),
        customdata=np.column_stack([
            daily_df["Fresh_coverage_pct"],
            daily_df["Fresh_pixels"],
        ]),
        hovertemplate=(
            "%{x|%Y-%m-%d}<br>Relative DNB-BRDF: %{y:.1f}%<br>"
            "Fresh coverage: %{customdata[0]:.1f}%<br>"
            "Fresh pixels: %{customdata[1]:.0f}<extra></extra>"
        ),
    ),
    row=1,
    col=1,
)

fig_trajectory.add_trace(
    go.Scatter(
        x=daily_df["Date"],
        y=daily_df["Relative_DNB_smoothed_pct"],
        mode="lines",
        name=f"Fresh {SMOOTHING_WINDOW} median",
        line=dict(color=COLORS["smoothed"], width=3),
    ),
    row=1,
    col=1,
)

fig_trajectory.add_trace(
    go.Scatter(
        x=daily_df["Date"],
        y=daily_df["Relative_gap_smoothed_pct"],
        mode="lines",
        name=f"Gap-filled {SMOOTHING_WINDOW} median",
        line=dict(color=COLORS["gap_filled"], width=2, dash="dot"),
    ),
    row=1,
    col=1,
)

fig_trajectory.add_hline(
    y=100,
    line_color="rgba(0,0,0,0.45)",
    line_dash="dash",
    row=1,
    col=1,
)

fig_trajectory.add_trace(
    go.Scatter(
        x=daily_df["Date"],
        y=daily_df["Fresh_coverage_pct"],
        mode="lines",
        name="Fresh coverage",
        line=dict(color=COLORS["coverage"], width=2),
        fill="tozeroy",
        fillcolor="rgba(44,160,44,0.12)",
    ),
    row=2,
    col=1,
)

fig_trajectory.add_trace(
    go.Scatter(
        x=daily_df["Date"],
        y=daily_df["Cloudy_pct"],
        mode="lines",
        name="Cloudy",
        line=dict(color=COLORS["cloud"], width=1.5),
    ),
    row=2,
    col=1,
)

fig_trajectory.add_hline(
    y=MIN_DAILY_COVERAGE,
    line_color=COLORS["event"],
    line_dash="dot",
    annotation_text=f"{MIN_DAILY_COVERAGE}% qualification threshold",
    annotation_position="top left",
    row=2,
    col=1,
)

add_haiyan_marker(fig_trajectory, row=1, col=1)
add_haiyan_marker(fig_trajectory, row=2, col=1)

fig_trajectory.add_annotation(
    x=EVENT_DATE,
    y=1.01,
    xref="x",
    yref="paper",
    text="Haiyan · 8 Nov 2013",
    showarrow=False,
    xanchor="left",
    font=dict(color=COLORS["event"], size=14),
)

fig_trajectory.update_yaxes(
    title_text="Relative nighttime lights (%)",
    row=1,
    col=1,
)
fig_trajectory.update_yaxes(
    title_text="Supported area (%)",
    range=[0, 100],
    row=2,
    col=1,
)
fig_trajectory.update_xaxes(title_text="Date", row=2, col=1)
fig_trajectory.update_layout(
    title=(
        "Reliability-qualified VNP46A2 trajectory around Haiyan"
        "<br><sup>Fresh DNB-BRDF is primary; gap-filled NTL is diagnostic only</sup>"
    )
)

apply_figure_style(fig_trajectory, height=820, top=125)
fig_trajectory.show()
export_figure(fig_trajectory, "fig_01_trajectory_and_observability")

NameError: name 'make_subplots' is not defined

## Pre-event anomalies and product explanations

This section does not model every acquisition variable. It isolates unusual pre-event DNB-BRDF dates and asks whether they coincide with low fresh coverage, cloud, older retrievals, or divergence from the gap-filled product. These diagnostics explain questionable observations without becoming a separate product audit.

In [2]:
pre_event = daily_df[
    daily_df["Days_from_event"].between(
        LONG_BASELINE_START,
        IMMEDIATE_BASELINE_END,
    )
    & daily_df["Qualified"]
].copy()

pre_event["Absolute_anomaly_pct"] = (
    pre_event["Relative_DNB_pct"] - 100
).abs()
pre_event["Observed_gap_difference_pct"] = (
    pre_event["Relative_DNB_pct"]
    - pre_event["Relative_gap_filled_pct"]
)

anomaly_columns = [
    "Date",
    "Relative_DNB_pct",
    "Relative_gap_filled_pct",
    "Observed_gap_difference_pct",
    "Fresh_coverage_pct",
    "Cloudy_pct",
    "Gap_filled_only_pct",
    "Retrieval_age_median",
    "Lunar_irradiance_median",
]

anomaly_dates_df = (
    pre_event
    .sort_values("Absolute_anomaly_pct", ascending=False)
    .head(10)[anomaly_columns]
    .reset_index(drop=True)
)

display(anomaly_dates_df.round(3))

if SAVE_OUTPUTS:
    anomaly_dates_df.to_csv(
        OUTPUT_TABLE_DIR / "pre_event_anomalous_dates.csv",
        index=False,
    )

fig_anomaly = make_subplots(
    rows=1,
    cols=2,
    horizontal_spacing=0.13,
    subplot_titles=[
        "Signal versus fresh coverage",
        "Observed versus gap-filled signal",
    ],
)

fig_anomaly.add_trace(
    go.Scatter(
        x=pre_event["Fresh_coverage_pct"],
        y=pre_event["Relative_DNB_pct"],
        mode="markers",
        marker=dict(
            size=9,
            color=pre_event["Cloudy_pct"],
            colorscale="Greys",
            cmin=0,
            cmax=100,
            colorbar=dict(title="Cloudy (%)", x=0.44),
            line=dict(color="white", width=0.5),
        ),
        customdata=np.column_stack([
            pre_event["Date"].dt.strftime("%Y-%m-%d"),
            pre_event["Retrieval_age_median"],
        ]),
        hovertemplate=(
            "%{customdata[0]}<br>Fresh coverage: %{x:.1f}%<br>"
            "Relative DNB-BRDF: %{y:.1f}%<br>"
            "Median retrieval age: %{customdata[1]:.1f} days"
            "<extra></extra>"
        ),
        showlegend=False,
    ),
    row=1,
    col=1,
)

fig_anomaly.add_trace(
    go.Scatter(
        x=pre_event["Relative_gap_filled_pct"],
        y=pre_event["Relative_DNB_pct"],
        mode="markers",
        marker=dict(
            size=9,
            color=pre_event["Gap_filled_only_pct"],
            colorscale="Oranges",
            colorbar=dict(title="Filled-only (%)", x=1.02),
            line=dict(color="white", width=0.5),
        ),
        customdata=pre_event["Date"].dt.strftime("%Y-%m-%d"),
        hovertemplate=(
            "%{customdata}<br>Gap-filled: %{x:.1f}%<br>"
            "Fresh DNB-BRDF: %{y:.1f}%<extra></extra>"
        ),
        showlegend=False,
    ),
    row=1,
    col=2,
)

comparison_values = pd.concat([
    pre_event["Relative_DNB_pct"],
    pre_event["Relative_gap_filled_pct"],
]).dropna()
comparison_min = max(0, float(comparison_values.quantile(0.01)))
comparison_max = float(comparison_values.quantile(0.99))

fig_anomaly.add_shape(
    type="line",
    x0=comparison_min,
    y0=comparison_min,
    x1=comparison_max,
    y1=comparison_max,
    line=dict(color="rgba(0,0,0,0.45)", dash="dash"),
    row=1,
    col=2,
)

fig_anomaly.update_xaxes(title_text="Fresh coverage (%)", row=1, col=1)
fig_anomaly.update_yaxes(
    title_text="Relative DNB-BRDF (%)", row=1, col=1
)
fig_anomaly.update_xaxes(
    title_text="Relative gap-filled NTL (%)", row=1, col=2
)
fig_anomaly.update_yaxes(
    title_text="Relative fresh DNB-BRDF (%)", row=1, col=2
)
fig_anomaly.update_layout(
    title=(
        "Pre-event signal anomalies and product dependence"
        "<br><sup>Descriptive diagnostics for questionable observations; not recovery metrics</sup>"
    )
)

apply_figure_style(fig_anomaly, width=1400, height=650, top=125, right=145)
fig_anomaly.show()
export_figure(fig_anomaly, "fig_02_pre_event_anomaly_diagnostics")

NameError: name 'daily_df' is not defined

## Spatial baseline reliability

The maps show the actual baseline radiance, the number of fresh observations supporting each pixel, and robust baseline variability. Only pixels satisfying all three baseline criteria enter the regional recovery profile.

In [ ]:
def display_grid(array):
    scale = min(1.0, MAX_MAP_DIMENSION / max(array.shape))
    rows = np.unique(
        np.linspace(0, array.shape[0] - 1, max(1, round(array.shape[0] * scale)))
        .round()
        .astype(int)
    )
    cols = np.unique(
        np.linspace(0, array.shape[1] - 1, max(1, round(array.shape[1] * scale)))
        .round()
        .astype(int)
    )

    if np.isclose(native_transform.b, 0) and np.isclose(native_transform.d, 0):
        x = native_transform.c + (cols + 0.5) * native_transform.a
        y = native_transform.f + (rows + 0.5) * native_transform.e
    else:
        x = cols
        y = rows

    return array[np.ix_(rows, cols)], x, y


baseline_map = np.where(eligible_mask, baseline_median, np.nan)
count_map = np.where(np.isfinite(baseline_median), baseline_count, np.nan)
variability_map = np.where(
    np.isfinite(baseline_median), baseline_robust_rsd, np.nan
)

map_arrays = [baseline_map, count_map, variability_map]
map_titles = [
    "Eligible baseline radiance",
    "Fresh baseline observations",
    "Baseline robust RSD",
]
map_coloraxes = ["coloraxis", "coloraxis2", "coloraxis3"]

fig_baseline_maps = make_subplots(
    rows=1,
    cols=3,
    horizontal_spacing=0.06,
    subplot_titles=map_titles,
)

for position, (array, coloraxis) in enumerate(
    zip(map_arrays, map_coloraxes), start=1
):
    displayed, x_coords, y_coords = display_grid(array)
    fig_baseline_maps.add_trace(
        go.Heatmap(
            x=x_coords,
            y=y_coords,
            z=displayed,
            coloraxis=coloraxis,
            zsmooth=False,
            hoverongaps=False,
            hovertemplate=(
                "X: %{x:.4f}<br>Y: %{y:.4f}<br>Value: %{z:.3f}"
                "<extra></extra>"
            ),
        ),
        row=1,
        col=position,
    )

baseline_upper = float(np.nanpercentile(baseline_map, 98))
variability_upper = max(
    0.1, float(np.nanpercentile(variability_map, 98))
)

fig_baseline_maps.update_layout(
    title=(
        "Spatial support for the pre-Haiyan baseline"
        "<br><sup>Maps retain the native raster geometry and use the actual VNP46A2 pixels</sup>"
    ),
    coloraxis=dict(
        colorscale="Viridis",
        cmin=0,
        cmax=baseline_upper,
        colorbar=dict(title="nW cm⁻² sr⁻¹", x=0.29, len=0.72),
    ),
    coloraxis2=dict(
        colorscale="Blues",
        cmin=0,
        cmax=len(baseline_dates),
        colorbar=dict(title="Observations", x=0.64, len=0.72),
    ),
    coloraxis3=dict(
        colorscale="Magma_r",
        cmin=0,
        cmax=variability_upper,
        colorbar=dict(title="Robust RSD", x=1.01, len=0.72),
    ),
)

apply_figure_style(
    fig_baseline_maps,
    width=1500,
    height=650,
    top=125,
    right=165,
    bottom=70,
)

for position in range(1, 4):
    xref = "x" if position == 1 else f"x{position}"
    fig_baseline_maps.update_xaxes(
        title_text="Longitude" if native_crs and native_crs.is_geographic else "X",
        showgrid=False,
        constrain="domain",
        row=1,
        col=position,
    )
    fig_baseline_maps.update_yaxes(
        title_text="Latitude" if position == 1 and native_crs and native_crs.is_geographic else None,
        showgrid=False,
        constrain="domain",
        scaleanchor=xref,
        scaleratio=1,
        row=1,
        col=position,
    )

fig_baseline_maps.show()
export_figure(fig_baseline_maps, "fig_03_spatial_baseline_reliability")

## Impact and interval-censored recovery metrics

Recovery thresholds are defined relative to the observed impact level:

\[
T_p = I + p(100-I),
\]

where \(I\) is the lowest reliability-qualified smoothed post-event value and \(p\) is 0.50 or 0.80. A threshold is considered sustained when another qualified observation at or above the threshold occurs within the persistence window. The crossing remains interval-censored because the exact recovery day may fall inside an unobserved interval.

In [ ]:
def sustained_crossing(profile, value_column, impact_date, threshold):
    candidates = (
        profile[
            profile["Qualified"]
            & profile["Date"].ge(impact_date)
            & profile[value_column].ge(threshold)
        ]
        .sort_values("Date")
    )

    for row in candidates.itertuples():
        confirmation = profile[
            profile["Qualified"]
            & profile["Date"].gt(row.Date)
            & profile["Date"].le(
                row.Date + pd.Timedelta(days=RECOVERY_PERSISTENCE_DAYS)
            )
            & profile[value_column].ge(threshold)
        ]

        if confirmation.empty:
            continue

        last_below = (
            profile[
                profile["Qualified"]
                & profile["Date"].ge(impact_date)
                & profile["Date"].lt(row.Date)
                & profile[value_column].lt(threshold)
            ]
            .sort_values("Date")
            .tail(1)
        )

        lower_date = (
            last_below.iloc[0]["Date"]
            if not last_below.empty
            else impact_date
        )

        return {
            "lower_date": lower_date,
            "upper_date": row.Date,
            "lower_day": int((lower_date - EVENT_DATE).days),
            "upper_day": int((row.Date - EVENT_DATE).days),
            "interval_days": int((row.Date - lower_date).days),
        }

    return None


def observed_deficit(profile, value_column):
    points = (
        profile[
            profile["Qualified"]
            & profile["Date"].ge(EVENT_DATE)
            & profile[value_column].notna()
        ][["Date", value_column]]
        .sort_values("Date")
    )

    total = 0.0
    supported_days = 0

    for first, second in zip(
        points.itertuples(index=False),
        points.iloc[1:].itertuples(index=False),
    ):
        gap_days = (second.Date - first.Date).days
        if gap_days > MAX_INTEGRATION_GAP_DAYS:
            continue

        first_deficit = max(0, 100 - getattr(first, value_column))
        second_deficit = max(0, 100 - getattr(second, value_column))
        total += (first_deficit + second_deficit) / 2 * gap_days
        supported_days += gap_days

    return total, supported_days


post_impact_window = daily_df[
    daily_df["Qualified"]
    & daily_df["Days_from_event"].between(0, IMPACT_WINDOW_DAYS)
    & daily_df["Relative_DNB_smoothed_pct"].notna()
]

impact_result = None
t50_result = None
t80_result = None
gap_impact_result = None
gap_t80_result = None

if not post_impact_window.empty:
    impact_row = post_impact_window.loc[
        post_impact_window["Relative_DNB_smoothed_pct"].idxmin()
    ]
    impact_date = impact_row["Date"]
    impact_level = float(impact_row["Relative_DNB_smoothed_pct"])
    impact_drop = 100 - impact_level

    t50_threshold = impact_level + 0.50 * impact_drop
    t80_threshold = impact_level + 0.80 * impact_drop

    t50_result = sustained_crossing(
        daily_df,
        "Relative_DNB_smoothed_pct",
        impact_date,
        t50_threshold,
    )
    t80_result = sustained_crossing(
        daily_df,
        "Relative_DNB_smoothed_pct",
        impact_date,
        t80_threshold,
    )
    cumulative_deficit, integrated_days = observed_deficit(
        daily_df,
        "Relative_DNB_smoothed_pct",
    )

    impact_result = {
        "date": impact_date,
        "day": int((impact_date - EVENT_DATE).days),
        "level": impact_level,
        "drop": impact_drop,
        "coverage": float(impact_row["Fresh_coverage_pct"]),
    }

gap_impact_window = daily_df[
    daily_df["Gap_available"]
    & daily_df["Days_from_event"].between(0, IMPACT_WINDOW_DAYS)
    & daily_df["Relative_gap_smoothed_pct"].notna()
]

if not gap_impact_window.empty:
    gap_impact_row = gap_impact_window.loc[
        gap_impact_window["Relative_gap_smoothed_pct"].idxmin()
    ]
    gap_impact_date = gap_impact_row["Date"]
    gap_impact_level = float(gap_impact_row["Relative_gap_smoothed_pct"])
    gap_impact_drop = 100 - gap_impact_level
    gap_t80_threshold = gap_impact_level + 0.80 * gap_impact_drop

    gap_profile = daily_df.copy()
    gap_profile["Qualified"] = gap_profile["Gap_available"]
    gap_t80_result = sustained_crossing(
        gap_profile,
        "Relative_gap_smoothed_pct",
        gap_impact_date,
        gap_t80_threshold,
    )
    gap_impact_result = {
        "date": gap_impact_date,
        "drop": gap_impact_drop,
    }

post_qualified_count = int(
    daily_df["Qualified"].where(daily_df["Date"].ge(EVENT_DATE), False).sum()
)

if impact_result is None:
    observation_status = "Not observable / post-event pending"
elif t80_result is None:
    observation_status = "Shock only"
elif t80_result["interval_days"] <= MAX_INTEGRATION_GAP_DAYS:
    observation_status = "Profile admissible"
else:
    observation_status = "Envelope only"

if impact_result is None and gap_impact_result is not None:
    product_status = "Gap-filled dependent"
elif impact_result is None:
    product_status = "Observed-only insufficient"
elif gap_impact_result is None:
    product_status = "Observed-only; gap comparison unavailable"
else:
    impact_difference = abs(
        impact_result["drop"] - gap_impact_result["drop"]
    )
    if t80_result is not None and gap_t80_result is not None:
        t80_difference = abs(
            t80_result["upper_day"] - gap_t80_result["upper_day"]
        )
    elif t80_result is None and gap_t80_result is None:
        t80_difference = 0
    else:
        t80_difference = np.inf

    product_status = (
        "Robust"
        if impact_difference <= 10 and t80_difference <= 14
        else "Gap-filled sensitive"
    )

metric_rows = []

if impact_result is None:
    metric_rows.extend([
        {
            "Metric": metric,
            "Estimate": np.nan,
            "Interval": "Pending",
            "Fresh_observations": post_qualified_count,
            "Support": observation_status,
            "Product_sensitivity": product_status,
        }
        for metric in ["Impact drop", "T50", "T80", "Cumulative deficit"]
    ])
else:
    metric_rows.append({
        "Metric": "Impact drop",
        "Estimate": f"{impact_result['drop']:.1f} percentage points",
        "Interval": impact_result["date"].strftime("%Y-%m-%d"),
        "Fresh_observations": post_qualified_count,
        "Support": observation_status,
        "Product_sensitivity": product_status,
    })

    for label, result in [("T50", t50_result), ("T80", t80_result)]:
        metric_rows.append({
            "Metric": label,
            "Estimate": (
                f"Day {result['upper_day']}"
                if result is not None
                else np.nan
            ),
            "Interval": (
                f"Day {result['lower_day']} to {result['upper_day']}"
                if result is not None
                else "Not observed"
            ),
            "Fresh_observations": post_qualified_count,
            "Support": observation_status,
            "Product_sensitivity": product_status,
        })

    metric_rows.append({
        "Metric": "Cumulative deficit",
        "Estimate": f"{cumulative_deficit:.1f} percentage-point days",
        "Interval": f"{integrated_days} observed/interpolable days",
        "Fresh_observations": post_qualified_count,
        "Support": observation_status,
        "Product_sensitivity": product_status,
    })

recovery_metrics_df = pd.DataFrame(metric_rows)
display(recovery_metrics_df)

if SAVE_OUTPUTS:
    recovery_metrics_df.to_csv(
        OUTPUT_TABLE_DIR / "recovery_metrics.csv", index=False
    )

if impact_result is not None:
    fig_recovery = go.Figure()

    fig_recovery.add_trace(
        go.Scatter(
            x=daily_df["Date"],
            y=daily_df["Relative_DNB_smoothed_pct"],
            mode="lines+markers",
            name="Fresh DNB-BRDF",
            line=dict(color=COLORS["smoothed"], width=3),
            marker=dict(size=6),
        )
    )
    fig_recovery.add_trace(
        go.Scatter(
            x=daily_df["Date"],
            y=daily_df["Relative_gap_smoothed_pct"],
            mode="lines",
            name="Gap-filled diagnostic",
            line=dict(color=COLORS["gap_filled"], width=2, dash="dot"),
        )
    )

    fig_recovery.add_hline(
        y=100,
        line_color="rgba(0,0,0,0.45)",
        line_dash="dash",
        annotation_text="Pixel-level baseline",
        annotation_position="top left",
    )

    for label, threshold, result, color in [
        ("T50", t50_threshold, t50_result, "#9467bd"),
        ("T80", t80_threshold, t80_result, "#2ca02c"),
    ]:
        fig_recovery.add_hline(
            y=threshold,
            line_color=color,
            line_dash="dot",
            annotation_text=f"{label} threshold ({threshold:.1f}%)",
            annotation_position="bottom right",
        )

        if result is not None:
            fig_recovery.add_vrect(
                x0=result["lower_date"],
                x1=result["upper_date"],
                fillcolor=color,
                opacity=0.16,
                line_width=0,
                annotation_text=f"{label} interval",
                annotation_position="top left",
            )

    fig_recovery.add_trace(
        go.Scatter(
            x=[impact_result["date"]],
            y=[impact_result["level"]],
            mode="markers+text",
            name="Observed impact",
            marker=dict(size=12, color="black", symbol="diamond"),
            text=[f"Impact −{impact_result['drop']:.1f} pp"],
            textposition="bottom right",
        )
    )

    add_haiyan_marker(fig_recovery)
    fig_recovery.add_annotation(
        x=EVENT_DATE,
        y=1.01,
        xref="x",
        yref="paper",
        text="Haiyan · 8 Nov 2013",
        showarrow=False,
        xanchor="left",
        font=dict(color=COLORS["event"], size=14),
    )

    fig_recovery.update_layout(
        title=(
            "Observed Haiyan shock and interval-censored recovery"
            "<br><sup>Shaded bands represent timing uncertainty between qualified observations</sup>"
        ),
        xaxis_title="Date",
        yaxis_title="Relative nighttime lights (%)",
    )
    apply_figure_style(fig_recovery, height=720, top=125)
    fig_recovery.show()
    export_figure(fig_recovery, "fig_04_shock_t50_t80")
else:
    display(Markdown(
        "**Recovery status.** No reliability-qualified post-event impact "
        "window is available yet. The metrics and recovery figure will be "
        "generated automatically after the post-event GeoTIFF stacks are added."
    ))

## Representative spatial snapshots

Dates are selected from the trajectory rather than manually chosen:

- **Typical pre-event:** qualified immediate-baseline date closest to 100% with strong coverage.
- **Observed impact:** lowest qualified smoothed value within 30 days after landfall.
- **T80 observation:** first sustained qualified observation above the T80 threshold.

The top row shows only fresh high-quality DNB-BRDF pixels. The bottom row shows the gap-filled product for the same dates. Blank areas in the top row are observational gaps, not zero radiance.

In [ ]:
immediate_pre = daily_df[
    daily_df["Days_from_event"].between(
        IMMEDIATE_BASELINE_START,
        IMMEDIATE_BASELINE_END,
    )
    & daily_df["Qualified"]
].copy()

snapshot_dates = []
snapshot_labels = []

if not immediate_pre.empty:
    immediate_pre["Selection_score"] = (
        (immediate_pre["Relative_DNB_pct"] - 100).abs()
        + 0.10 * (100 - immediate_pre["Fresh_coverage_pct"])
    )
    typical_pre_date = immediate_pre.loc[
        immediate_pre["Selection_score"].idxmin(), "Date"
    ]
    snapshot_dates.append(typical_pre_date)
    snapshot_labels.append("Typical pre-event")

if impact_result is not None:
    snapshot_dates.append(impact_result["date"])
    snapshot_labels.append("Observed impact")

if t80_result is not None:
    snapshot_dates.append(t80_result["upper_date"])
    snapshot_labels.append("T80 observation")

# Preserve order while removing repeated dates.
unique_snapshots = []
for date, label in zip(snapshot_dates, snapshot_labels):
    if date not in [item[0] for item in unique_snapshots]:
        unique_snapshots.append((date, label))

if unique_snapshots:
    snapshot_arrays = []

    with ExitStack() as stack:
        sources = {
            str(path): stack.enter_context(rasterio.open(path))
            for path in INPUT_FILES
        }

        for date, label in unique_snapshots:
            dnb = read_band(
                sources, band_lookup, date,
                "DNB_BRDF_Corrected_NTL", radiance_scale,
            )
            gap = read_band(
                sources, band_lookup, date,
                "Gap_Filled_DNB_BRDF_Corrected_NTL", radiance_scale,
            )
            mqf = read_band(
                sources, band_lookup, date,
                "Mandatory_Quality_Flag", radiance_scale,
            )
            snow = read_band(
                sources, band_lookup, date,
                "Snow_Flag", radiance_scale,
            )
            cloud_qf = read_band(
                sources, band_lookup, date,
                "QF_Cloud_Mask", radiance_scale,
            )
            cloud = decode_cloud_mask(cloud_qf)
            land = cloud["valid"] & np.isin(cloud["land_water"], [0, 1, 5])

            fresh = (
                eligible_mask
                & land
                & np.isfinite(dnb)
                & np.isfinite(mqf)
                & (np.rint(mqf) == 0)
                & np.isfinite(snow)
                & (np.rint(snow) == 0)
            )
            gap_valid = eligible_mask & land & np.isfinite(gap)

            observed_relative = np.divide(
                dnb,
                baseline_median,
                out=np.full_like(baseline_median, np.nan),
                where=fresh,
            ) * 100
            gap_relative = np.divide(
                gap,
                baseline_median,
                out=np.full_like(baseline_median, np.nan),
                where=gap_valid,
            ) * 100

            daily_row = daily_df.loc[daily_df["Date"].eq(date)].iloc[0]
            snapshot_arrays.append({
                "date": date,
                "label": label,
                "observed": observed_relative,
                "gap": gap_relative,
                "coverage": daily_row["Fresh_coverage_pct"],
            })

    subplot_titles = []
    for item in snapshot_arrays:
        title = (
            f"{item['label']} · {item['date']:%Y-%m-%d}"
            f"<br><sup>Fresh coverage: {item['coverage']:.1f}%</sup>"
        )
        subplot_titles.append(title)
    subplot_titles += [item["label"] for item in snapshot_arrays]

    column_count = len(snapshot_arrays)
    fig_snapshots = make_subplots(
        rows=2,
        cols=column_count,
        horizontal_spacing=0.04,
        vertical_spacing=0.12,
        subplot_titles=subplot_titles,
    )

    for column, item in enumerate(snapshot_arrays, start=1):
        for row, key in [(1, "observed"), (2, "gap")]:
            displayed, x_coords, y_coords = display_grid(item[key])
            fig_snapshots.add_trace(
                go.Heatmap(
                    x=x_coords,
                    y=y_coords,
                    z=displayed,
                    coloraxis="coloraxis",
                    zsmooth=False,
                    hoverongaps=False,
                    hovertemplate=(
                        "X: %{x:.4f}<br>Y: %{y:.4f}<br>"
                        "Relative NTL: %{z:.1f}%<extra></extra>"
                    ),
                ),
                row=row,
                col=column,
            )

    fig_snapshots.update_layout(
        title=(
            "Representative spatial nighttime lights around Haiyan"
            "<br><sup>Top: fresh DNB-BRDF observations · Bottom: gap-filled diagnostic</sup>"
        ),
        coloraxis=dict(
            colorscale="RdBu",
            cmin=0,
            cmax=160,
            cmid=100,
            colorbar=dict(title="Relative NTL (%)", x=1.01),
        ),
    )

    apply_figure_style(
        fig_snapshots,
        width=max(850, 480 * column_count),
        height=930,
        top=135,
        right=145,
        bottom=70,
    )

    for position in range(1, 2 * column_count + 1):
        row = (position - 1) // column_count + 1
        col = (position - 1) % column_count + 1
        xref = "x" if position == 1 else f"x{position}"
        fig_snapshots.update_xaxes(
            title_text=(
                "Longitude"
                if row == 2 and native_crs and native_crs.is_geographic
                else None
            ),
            showgrid=False,
            constrain="domain",
            row=row,
            col=col,
        )
        fig_snapshots.update_yaxes(
            title_text=(
                "Fresh DNB-BRDF"
                if row == 1 and col == 1
                else "Gap-filled"
                if row == 2 and col == 1
                else None
            ),
            showgrid=False,
            constrain="domain",
            scaleanchor=xref,
            scaleratio=1,
            row=row,
            col=col,
        )

    fig_snapshots.show()
    export_figure(fig_snapshots, "fig_05_representative_spatial_snapshots")
else:
    display(Markdown(
        "**Spatial snapshot status.** No qualified immediate pre-event date "
        "is available under the current thresholds."
    ))

## Interpretation and next decision

The notebook should support one of four conclusions:

| Observation status | Permitted interpretation |
|---|---|
| Profile admissible | Report shock, interval-censored T50/T80, and cumulative deficit. |
| Envelope only | Report shock and recovery timing range; do not report an exact recovery date. |
| Shock only | Report observed impact; do not infer recovery duration. |
| Not observable / post-event pending | Do not infer impact or recovery. |

A smooth gap-filled trajectory does not upgrade the observational status. It is evidence of product continuity, not direct evidence of recovery.

After the regional prototype is complete, the same workflow can be stratified by GHSL settlement class and applied to 1×1, 3×3, and 5×5 local extents. VNP46A1 VZA sensitivity should be introduced at that finer scale only if it materially changes the recovery metrics.

In [ ]:
available_post = daily_df[
    daily_df["File_available"] & daily_df["Date"].ge(EVENT_DATE)
]
qualified_post = available_post[available_post["Qualified"]]

summary_text = (
    f"**Overall result.** The baseline retains **{eligible_count:,}** stable, "
    f"non-zero pixels from **{len(baseline_dates)}** available pre-event dates. "
    f"The full calendar currently contains **{len(available_dates):,}/"
    f"{len(expected_dates):,}** available dates and **{len(qualified_post):,}** "
    f"reliability-qualified post-event dates. The recovery profile is classified "
    f"as **{observation_status}**, with product sensitivity classified as "
    f"**{product_status}**."
)

if impact_result is None:
    summary_text += (
        " Impact and recovery remain pending; missing post-event observations "
        "must not be interpreted as sustained darkness."
    )
else:
    summary_text += (
        f" The observed impact is **{impact_result['drop']:.1f} percentage "
        f"points below baseline** on **{impact_result['date']:%Y-%m-%d}**, "
        f"supported by **{impact_result['coverage']:.1f}%** fresh coverage."
    )

    if t80_result is not None:
        summary_text += (
            f" T80 is bounded between **day {t80_result['lower_day']} and "
            f"day {t80_result['upper_day']}** after Haiyan; this interval, "
            "not a single exact date, is the defensible recovery estimate."
        )
    else:
        summary_text += (
            " T80 is not observed with sufficient persistence, so no recovery "
            "duration should be reported."
        )

display(Markdown(summary_text))